Training Bicep Model

In [51]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, accuracy_score, f1_score, recall_score, confusion_matrix
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
import pickle
# import warnings
# warnings.filterwarnings('ignore')

In [52]:
# Import train and test datasets
train_biceps=pd.read_csv('Datasets/train_biceps.csv')
test_biceps=pd.read_csv('Datasets/test_bicep.csv')

In [53]:
train_biceps.loc[train_biceps["label"] == "C", "label"] = 0
train_biceps.loc[train_biceps["label"] == "L", "label"] = 1
test_biceps.loc[test_biceps["label"] == "C", "label"] = 0
test_biceps.loc[test_biceps["label"] == "L", "label"] = 1
sc = StandardScaler()
x = train_biceps.drop("label", axis = 1)
x = pd.DataFrame(sc.fit_transform(x))
y = train_biceps["label"].astype('int')
x.head()

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
0,0.971653,1.142182,-0.550735,-0.097000,-0.623713,-0.178082,0.728639,-3.415250,0.349215,-0.161738,...,0.805633,-1.610907,-1.855656,-0.009522,1.258496,-1.730899,-1.082622,0.068957,-1.260549,-0.792337
1,0.796054,0.836857,-0.497872,-0.073777,-0.628932,-0.200357,0.770301,-3.514380,0.141478,-0.370173,...,0.854093,-1.615338,-1.675443,0.162899,1.228429,-2.515633,-1.028284,0.212495,-1.230049,-1.290633
2,0.763438,0.762989,-0.568673,-0.002869,-0.644330,-0.212601,0.786812,-3.323405,0.103939,-0.367530,...,0.786174,-1.615969,-1.655221,0.226325,1.225639,-2.675350,-1.011337,0.271010,-1.227198,-1.403055
3,0.742864,0.754111,-0.632002,0.059946,-0.663456,-0.217468,0.764685,-3.092643,0.092610,-0.363672,...,0.738584,-1.614220,-1.638127,0.281960,1.198155,-2.553494,-0.984588,0.324522,-1.199558,-1.310745
4,0.727373,0.714976,-0.777586,0.123915,-0.670168,-0.219923,0.753538,-2.823182,0.092610,-0.365096,...,0.741803,-1.611346,-1.601547,0.318932,1.184691,-2.289814,-0.957505,0.348025,-1.186042,-1.138229


In [ ]:
with open ('Models/bicep_curl/input_scaler.pkl', 'wb') as f:
    pickle.dump(sc)


In [54]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1234)
y_train

9465     1
8833     0
6190     0
7645     0
13890    1
        ..
11468    1
7221     1
1318     1
8915     1
11055    1
Name: label, Length: 12297, dtype: int64

In [55]:
algorithms =[
         ("SVC", SVC(probability=True)),
         ('KNN',KNeighborsClassifier()),
         ]
models = {}
final_results = []

for name, model in algorithms:
    trained_model = model.fit(X_train, y_train)
    models[name] = trained_model

    # Evaluate model
    model_results = model.predict(X_test)

    p_score = precision_score(y_test, model_results, average=None, labels=[0, 1])
    a_score = accuracy_score(y_test, model_results)
    r_score = recall_score(y_test, model_results, average=None, labels=[0, 1])
    f1_score_result = f1_score(y_test, model_results, average=None, labels=[0, 1])
    cm = confusion_matrix(y_test, model_results, labels=[0, 1])
    final_results.append(( name,  p_score, a_score, r_score, f1_score_result, cm))

# Sort results by F1 score
final_results.sort(key=lambda k: sum(k[4]), reverse=True)
pd.DataFrame(final_results, columns=["Model", "Precision Score", "Accuracy score", "Recall Score", "F1 score", "Confusion Matrix"])

,Model,Precision Score,Accuracy score,Recall Score,F1 score,Confusion Matrix
0,KNN,"[0.9970291146761735, 0.9992816091954023]",0.998049,"[0.9994044073853484, 0.9964183381088825]","[0.9982153480071386, 0.9978479196556671]","[[1678, 1], [5, 1391]]"
1,SVC,"[0.9970184853905785, 0.9949928469241774]",0.996098,"[0.9958308516974389, 0.9964183381088825]","[0.9964243146603099, 0.9957050823192556]","[[1672, 7], [5, 1391]]"


In [56]:
# --- Neural Network Model ---
nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # binary classification
])

nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train NN
history = nn_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

# Predictions
nn_preds = (nn_model.predict(X_test) > 0.5).astype(int).flatten()

p_score = precision_score(y_test, nn_preds, average=None, labels=[0, 1])
a_score = accuracy_score(y_test, nn_preds)
r_score = recall_score(y_test, nn_preds, average=None, labels=[0, 1])
f1_score_result = f1_score(y_test, nn_preds, average=None, labels=[0, 1])
cm = confusion_matrix(y_test, nn_preds, labels=[0, 1])

final_results.append(("NeuralNet", p_score, a_score, r_score, f1_score_result, cm))

/Users/sarthakjain/miniforge3/envs/major-project/lib/python3.10/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 889us/step


In [57]:
final_results.sort(key=lambda k: sum(k[4]), reverse=True)
results_df = pd.DataFrame(final_results,
                          columns=["Model", "Precision Score", "Accuracy score",
                                   "Recall Score", "F1 score", "Confusion Matrix"])
results_df

,Model,Precision Score,Accuracy score,Recall Score,F1 score,Confusion Matrix
0,NeuralNet,"[0.99880810488677, 0.9978525411596277]",0.998374,"[0.9982132221560452, 0.998567335243553]","[0.9985105749180816, 0.9982098102398854]","[[1676, 3], [2, 1394]]"
1,KNN,"[0.9970291146761735, 0.9992816091954023]",0.998049,"[0.9994044073853484, 0.9964183381088825]","[0.9982153480071386, 0.9978479196556671]","[[1678, 1], [5, 1391]]"
2,SVC,"[0.9970184853905785, 0.9949928469241774]",0.996098,"[0.9958308516974389, 0.9964183381088825]","[0.9964243146603099, 0.9957050823192556]","[[1672, 7], [5, 1391]]"


In [ ]:
# Dump the best model here, will do later according to the frontend api call
knn_model = models['KNN']
with open("./model/bicep_curl.pkl", "wb") as f:
    pickle.dump(knn_model, f)

Squat Model Training


In [59]:
train_squat=pd.read_csv('Datasets/train_squat.csv')
test_squat=pd.read_csv('Datasets/test_squat.csv')


In [60]:
train_squat.head()

,label,nose_x,nose_y,nose_z,nose_v,left_shoulder_x,left_shoulder_y,left_shoulder_z,left_shoulder_v,right_shoulder_x,...,right_knee_z,right_knee_v,left_ankle_x,left_ankle_y,left_ankle_z,left_ankle_v,right_ankle_x,right_ankle_y,right_ankle_z,right_ankle_v
0,down,0.600248,0.433268,-0.006637,0.999949,0.650759,0.517787,0.041632,0.994121,0.571711,...,-0.133012,0.976462,0.659328,0.836607,-0.294549,0.994219,0.575143,0.814380,-0.157936,0.966270
1,down,0.600160,0.449894,-0.008615,0.999966,0.651127,0.535333,0.036240,0.994432,0.571456,...,-0.115595,0.974204,0.659568,0.831231,-0.279601,0.993431,0.576560,0.812989,-0.135235,0.959162
2,down,0.599425,0.466234,-0.047636,0.999957,0.651464,0.550511,-0.021722,0.994265,0.571824,...,-0.084128,0.949108,0.659044,0.828410,-0.202085,0.987354,0.575949,0.813362,-0.053549,0.928765
3,down,0.597382,0.485779,-0.057994,0.999926,0.651386,0.571049,-0.023268,0.994848,0.572379,...,-0.069531,0.921725,0.658455,0.832319,-0.168358,0.982599,0.576511,0.813286,-0.058735,0.904242
4,down,0.594474,0.438586,-0.041261,0.999931,0.650657,0.518908,-0.003251,0.993209,0.571241,...,-0.119639,0.969358,0.658565,0.836644,-0.274596,0.994046,0.576144,0.815093,-0.117950,0.965770


In [61]:
from sklearn.linear_model import LogisticRegression

There will be *9 keypoints* which will be extract from mediapipe in order to train or detect a correct form of a squat:
- "NOSE",
- "LEFT_SHOULDER",
- "RIGHT_SHOULDER",
- "LEFT_HIP",
- "RIGHT_HIP",
- "LEFT_KNEE",
- "RIGHT_KNEE",
- "LEFT_ANKLE",
- "RIGHT_ANKLE"

In [62]:
train_squat.loc[train_squat["label"] == "down", "label"] = 0
train_squat.loc[train_squat["label"] == "up", "label"] = 1
test_squat.loc[test_squat["label"] == "down", "label"] = 0
test_squat.loc[test_squat["label"] == "up", "label"] = 1




In [63]:
# Extract features and class
X_train = train_squat.drop("label", axis=1) # features
X_test=test_squat.drop("label", axis=1)
y_train = train_squat["label"].astype("int")
y_test=test_squat["label"].astype("int")
sc = StandardScaler()
X_train = pd.DataFrame(sc.fit_transform(X_train))
X_test = pd.DataFrame(sc.fit_transform(X_test))

In [64]:
algorithms =[("LR", LogisticRegression()),
         ("SVC", SVC(probability=True)),
         ('KNN',KNeighborsClassifier()),
]

models = {}
final_results = []

for name, model in algorithms:
    trained_model = model.fit(X_train, y_train)
    models[name] = trained_model

    # Evaluate model
    model_results = model.predict(X_test)

    p_score = precision_score(y_test, model_results, average=None, labels=[0, 1])
    a_score = accuracy_score(y_test, model_results)
    r_score = recall_score(y_test, model_results, average=None, labels=[0, 1])
    f1_score_result = f1_score(y_test, model_results, average=None, labels=[0, 1])
    cm = confusion_matrix(y_test, model_results, labels=[0, 1])
    final_results.append(( name,  p_score, a_score, r_score, f1_score_result, cm))

# Sort results by F1 score
final_results.sort(key=lambda k: sum(k[4]), reverse=True)
pd.DataFrame(final_results, columns=["Model", "Precision Score", "Accuracy score", "Recall Score", "F1 score", "Confusion Matrix"])


,Model,Precision Score,Accuracy score,Recall Score,F1 score,Confusion Matrix
0,LR,"[0.9930555555555556, 0.997624703087886]",0.995311,"[0.9976744186046511, 0.9929078014184397]","[0.9953596287703016, 0.995260663507109]","[[429, 1], [3, 420]]"
1,SVC,"[0.9930555555555556, 0.997624703087886]",0.995311,"[0.9976744186046511, 0.9929078014184397]","[0.9953596287703016, 0.995260663507109]","[[429, 1], [3, 420]]"
2,KNN,"[0.988479262672811, 0.9976133651551312]",0.992966,"[0.9976744186046511, 0.9881796690307328]","[0.9930555555555556, 0.9928741092636579]","[[429, 1], [5, 418]]"


In [65]:
# Dump the model into a model pickle file. 

Plank Model Training

In [66]:
train_plank=pd.read_csv('Datasets/train_plank.csv')
test_plank=pd.read_csv('Datasets/test_plank.csv')

In [67]:
train_plank.loc[train_plank["label"] == "C", "label"] = 0
train_plank.loc[train_plank["label"] == "H", "label"] = 1
train_plank.loc[train_plank["label"] == "L", "label"] = 2
test_plank.loc[test_plank["label"] == "C", "label"] = 0
test_plank.loc[test_plank["label"] == "H", "label"] = 1
test_plank.loc[test_plank["label"] == "L", "label"] = 2
train_plank['label'].value_counts()

label
0    9904
2    9546
1    9070
Name: count, dtype: int64

In [68]:
X_train = train_plank.drop("label", axis=1)
y_train = train_plank["label"].astype("int")
X_test=test_plank.drop('label', axis=1)
y_test = test_plank["label"].astype("int")


In [69]:
sc = StandardScaler()
X_train = pd.DataFrame(sc.fit_transform(X_train))
X_test= pd.DataFrame(sc.fit_transform(X_test))


In [70]:
algorithms =[("LR", LogisticRegression()),
         ("SVC", SVC(probability=True)),
         ('KNN',KNeighborsClassifier()),
        ]

models = {}
final_results = []

for name, model in algorithms:
    trained_model = model.fit(X_train, y_train)
    models[name] = trained_model

    # Evaluate model
    model_results = model.predict(X_test)

    p_score = precision_score(y_test, model_results, average=None, labels=[0, 1, 2])
    a_score = accuracy_score(y_test, model_results)
    r_score = recall_score(y_test, model_results, average=None, labels=[0, 1, 2])
    f1_score_result = f1_score(y_test, model_results, average=None, labels=[0, 1, 2])
    cm = confusion_matrix(y_test, model_results, labels=[0, 1, 2])
    final_results.append(( name,  p_score, a_score, r_score, f1_score_result, cm))

# Sort results by F1 score
final_results.sort(key=lambda k: sum(k[4]), reverse=True)

pd.DataFrame(final_results, columns=["Model", "Precision Score", "Accuracy score", "Recall Score", "F1 score", "Confusion Matrix"])



,Model,Precision Score,Accuracy score,Recall Score,F1 score,Confusion Matrix
0,SVC,"[0.9873417721518988, 1.0, 1.0]",0.995775,"[1.0, 0.991701244813278, 0.9957446808510638]","[0.9936305732484076, 0.9958333333333333, 0.997...","[[234, 0, 0], [2, 239, 0], [1, 0, 234]]"
1,KNN,"[0.9767441860465116, 0.92578125, 0.97907949790...",0.959155,"[0.8974358974358975, 0.983402489626556, 0.9957...","[0.9354120267260579, 0.9537223340040242, 0.987...","[[210, 19, 5], [4, 237, 0], [1, 0, 234]]"
2,LR,"[0.8863636363636364, 1.0, 1.0]",0.957746,"[1.0, 0.9294605809128631, 0.9446808510638298]","[0.9397590361445783, 0.9634408602150538, 0.971...","[[234, 0, 0], [17, 224, 0], [13, 0, 222]]"


In [71]:
# Dump model into a pickle file

Push ups Model Training

In [72]:
train_pushups=pd.read_csv('Datasets/train_push_ups.csv')
test_pushups=pd.read_csv('Datasets/test_push_ups.csv')


In [73]:
train_pushups.head()

,Shoulder_Angle,Elbow_Angle,Hip_Angle,Knee_Angle,Ankle_Angle,Shoulder_Ground_Angle,Elbow_Ground_Angle,Hip_Ground_Angle,Knee_Ground_Angle,Ankle_Ground_Angle,Form_Label
0,-0.982157,-0.452735,0.476274,0.321954,-0.315467,-0.007292,0.008978,0.274816,0.268857,0.272877,1
1,-0.688277,-0.054853,0.535327,0.351853,-0.085543,-0.007292,0.008978,0.274816,0.268857,0.272877,1
2,-1.140400,-2.128048,-2.218088,1.113355,-1.511977,-2.802477,-2.247337,-3.250256,-3.303900,0.644869,0
3,0.478877,0.159451,0.463670,0.468720,0.626489,-0.007292,0.008978,0.274816,0.268857,0.272877,1
4,0.350325,-0.240064,0.238479,-1.518489,-0.631264,-0.007292,0.008978,0.274816,0.268857,0.272877,1


In [74]:
X_train=train_pushups.drop("Form_Label", axis=1)
y_train=train_pushups['Form_Label']
X_test=test_pushups.drop("Form_Label", axis=1)
y_test=test_pushups['Form_Label']

In [75]:
algorithms =[("LR", LogisticRegression()),
         ("SVC", SVC(probability=True)),
         ('KNN',KNeighborsClassifier()),
        ]

models = {}
final_results = []

for name, model in algorithms:
    trained_model = model.fit(X_train, y_train)
    models[name] = trained_model

    # Evaluate model
    model_results = model.predict(X_test)

    p_score = precision_score(y_test, model_results, average=None, labels=[0, 1])
    a_score = accuracy_score(y_test, model_results)
    r_score = recall_score(y_test, model_results, average=None, labels=[0, 1])
    f1_score_result = f1_score(y_test, model_results, average=None, labels=[0, 1])
    cm = confusion_matrix(y_test, model_results, labels=[0, 1])
    final_results.append(( name,  p_score, a_score, r_score, f1_score_result, cm))

# Sort results by F1 score
final_results.sort(key=lambda k: sum(k[4]), reverse=True)

pd.DataFrame(final_results, columns=["Model", "Precision Score", "Accuracy score", "Recall Score", "F1 score", "Confusion Matrix"])

,Model,Precision Score,Accuracy score,Recall Score,F1 score,Confusion Matrix
0,SVC,"[1.0, 1.0]",1.000000,"[1.0, 1.0]","[1.0, 1.0]","[[391, 0], [0, 1953]]"
1,KNN,"[1.0, 0.9984662576687117]",0.998720,"[0.9923273657289002, 1.0]","[0.9961489088575096, 0.9992325402916347]","[[388, 3], [0, 1953]]"
2,LR,"[0.96, 0.9608717186726102]",0.960751,"[0.7979539641943734, 0.9933435739887353]","[0.8715083798882681, 0.9768378650553877]","[[312, 79], [13, 1940]]"


In [ ]:
# Dump this model into a pickle file